In [16]:
import numpy as np
import pandas as pd

In [17]:
def ler_malha(caminho_arquivo: str):
    with open(caminho_arquivo, "r") as arquivo:
        primeira_linha = arquivo.readline().split()
        numero_nos = int(primeira_linha[0])
        numero_elementos = int(primeira_linha[1])
        numero_bordas = int(primeira_linha[2])

        coordenadas = []
        for _ in range(numero_nos):
            linha = arquivo.readline().split()
            coordenadas.append([float(linha[0]), float(linha[1])])

        elementos = []
        for _ in range(numero_elementos):
            linha = arquivo.readline().split()
            elementos.append([int(valor) - 1 for valor in linha])

        bordas = []
        for _ in range(numero_bordas):
            linha = arquivo.readline().split()
            bordas.append([int(valor) - 1 for valor in linha])

    return (
        np.array(coordenadas),
        np.array(elementos, dtype=int),
        bordas,
    )


coordenadas, elementos, bordas = ler_malha("malha.dat")

print("Número de nós:", coordenadas.shape[0])
print("Número de elementos:", elementos.shape[0])
print("Número de regiões de borda:", len(bordas))

Número de nós: 3866
Número de elementos: 3386
Número de regiões de borda: 2


Queremos implementar uma rotina para calcular integrais do tipo

$$
\int_\Omega f(x)\,dx.
$$

Como a malha é formada por elementos quadrilaterais, decompomos a integral como soma das integrais em cada elemento:

$$
\int_\Omega f(x)\,dx
=
\sum_{K\in\tau_h}\int_K f(x)\,dx.
$$

Cada elemento físico $K$ é obtido a partir de um elemento quadrado de referência $\hat K$ por uma transformação isoparamétrica.

In [28]:
def funcoes_forma_bilineares(xi: float, eta: float) -> np.ndarray:
    N1 = 0.25 * (1.0 - xi) * (1.0 - eta)
    N2 = 0.25 * (1.0 + xi) * (1.0 - eta)
    N3 = 0.25 * (1.0 + xi) * (1.0 + eta)
    N4 = 0.25 * (1.0 - xi) * (1.0 + eta)

    return np.array([N1, N2, N3, N4])


def derivadas_funcoes_forma_bilineares(xi: float, eta: float):
    dN_dxi = np.array([
        -0.25 * (1.0 - eta),
         0.25 * (1.0 - eta),
         0.25 * (1.0 + eta),
        -0.25 * (1.0 + eta),
    ])

    dN_deta = np.array([
        -0.25 * (1.0 - xi),
        -0.25 * (1.0 + xi),
         0.25 * (1.0 + xi),
         0.25 * (1.0 - xi),
    ])

    return dN_dxi, dN_deta

In [29]:
def avaliar_transformacao(xi: float, eta: float, vertices: np.ndarray):
    N = funcoes_forma_bilineares(xi, eta)

    x = N @ vertices[:, 0]
    y = N @ vertices[:, 1]

    return x, y


def calcular_jacobiano(xi: float, eta: float, vertices: np.ndarray):
    dN_dxi, dN_deta = derivadas_funcoes_forma_bilineares(xi, eta)

    dx_dxi = dN_dxi @ vertices[:, 0]
    dx_deta = dN_deta @ vertices[:, 0]

    dy_dxi = dN_dxi @ vertices[:, 1]
    dy_deta = dN_deta @ vertices[:, 1]

    jacobiano = np.array([
        [dx_dxi, dx_deta],
        [dy_dxi, dy_deta],
    ])

    return jacobiano

Para integrar em um elemento físico $K$, fazemos a mudança de variáveis

$$
\int_K f(x,y)\,dx\,dy
=
\int_{\hat K}
f(F_K(\xi,\eta))
\left|\det J_K(\xi,\eta)\right|
\,d\xi\,d\eta.
$$

Como $\hat K=[-1,1]\times[-1,1]$, usamos uma quadratura de Gauss produto com $3$ pontos em cada direção.

Assim,

$$
\int_{\hat K} g(\xi,\eta)\,d\xi\,d\eta
\approx
\sum_{i=1}^{3}\sum_{j=1}^{3}
w_iw_j g(\xi_i,\eta_j).
$$

Aplicando isso em cada elemento, obtemos

$$
\int_\Omega f(x,y)\,dx\,dy
\approx
\sum_{K\in\tau_h}
\sum_{i=1}^{3}\sum_{j=1}^{3}
w_iw_j
f(F_K(\xi_i,\eta_j))
\left|\det J_K(\xi_i,\eta_j)\right|.
$$

In [30]:
def integrar_na_malha(funcao, coordenadas: np.ndarray, elementos: np.ndarray) -> float:
    pontos_gauss, pesos_gauss = np.polynomial.legendre.leggauss(3)

    integral_total = 0.0

    for elemento in elementos:
        vertices = coordenadas[elemento]

        integral_elemento = 0.0

        for xi, peso_xi in zip(pontos_gauss, pesos_gauss):
            for eta, peso_eta in zip(pontos_gauss, pesos_gauss):
                x, y = avaliar_transformacao(xi, eta, vertices)
                jacobiano = calcular_jacobiano(xi, eta, vertices)

                det_jacobiano = np.linalg.det(jacobiano)

                integral_elemento += (
                    peso_xi
                    * peso_eta
                    * funcao(x, y)
                    * abs(det_jacobiano)
                )

        integral_total += integral_elemento

    return integral_total

In [31]:
funcoes_teste = {
    "1": lambda x, y: 1.0,
    "sen(x1)": lambda x, y: np.sin(x),
    "sen(x1) sen(x2)": lambda x, y: np.sin(x) * np.sin(y),
    "sen(x1) cos(x2)": lambda x, y: np.sin(x) * np.cos(y),
}

valores_referencia = {
    "1": 4606.110,
    "sen(x1)": -29.90879,
    "sen(x1) sen(x2)": 7.279703,
    "sen(x1) cos(x2)": -11.67626,
}

resultados = []

for nome, funcao in funcoes_teste.items():
    valor_calculado = integrar_na_malha(funcao, coordenadas, elementos)
    valor_referencia = valores_referencia[nome]

    resultados.append({
        "integral": nome,
        "valor_calculado": valor_calculado,
        "valor_referencia": valor_referencia,
        "erro_absoluto": abs(valor_calculado - valor_referencia),
    })

tabela_resultados = pd.DataFrame(resultados)
tabela_resultados

,integral,valor_calculado,valor_referencia,erro_absoluto
0,1,4606.110419,4606.110000,4.193666e-04
1,sen(x1),-29.908788,-29.908790,2.465640e-06
2,sen(x1) sen(x2),7.279703,7.279703,4.006779e-07
3,sen(x1) cos(x2),-11.676261,-11.676260,9.350689e-07


Isso confirma que:

1. a leitura da malha foi feita corretamente;
2. a conectividade dos elementos foi interpretada corretamente;
3. a transformação isoparamétrica está correta;
4. o cálculo do determinante do jacobiano está correto;
5. a quadratura de Gauss $3\times 3$ está funcionando como esperado.

In [32]:
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

Agora resolvemos o problema variacional: encontrar $u_h\in V_g$ tal que

$$
\int_\Omega \nabla u_h(x)\cdot \nabla v_h(x)\,dx
=
\int_\Omega f(x)v_h(x)\,dx,
\qquad
\forall v_h\in V_0,
$$

onde

$$
V_g=\{v_h\in V_h;\ v_h(x)=g(x),\ x\in\partial\Omega\},
$$

e

$$
V_0=\{v_h\in V_h;\ v_h(x)=0,\ x\in\partial\Omega\}.
$$

A função fonte é

$$
f(x)=
\sin\left(\frac{\pi x_1}{15}\right)
\sin\left(\frac{\pi x_2}{15}\right),
$$

e a condição de contorno é

$$
g(x)=
\begin{cases}
15, & \text{se } x \text{ está sobre a região de borda 1},\\
20, & \text{se } x \text{ está sobre a região de borda 2}.
\end{cases}
$$

Definimos a função fonte por

$$
f(x_1,x_2)
=
\sin\left(\frac{\pi x_1}{15}\right)
\sin\left(\frac{\pi x_2}{15}\right).
$$

A condição de Dirichlet é imposta nos nós de borda.

Nos nós da região de borda 1, impomos

$$
u_h=15.
$$

Nos nós da região de borda 2, impomos

$$
u_h=20.
$$

Assim, os graus de liberdade associados aos nós de borda são fixados, e o sistema linear será resolvido apenas para os nós interiores.

In [38]:
def fonte(x: float, y: float) -> float:
    return np.sin(np.pi * x / 15.0) * np.sin(np.pi * y / 15.0)


def construir_valores_contorno(coordenadas: np.ndarray, bordas: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    numero_nos = coordenadas.shape[0]

    valores_contorno = np.zeros(numero_nos)
    nos_contorno = np.array([], dtype=int)

    # Região de borda 1: g = 15
    valores_contorno[bordas[0]] = 15.0

    # Região de borda 2: g = 20
    valores_contorno[bordas[1]] = 20.0

    nos_contorno = np.unique(np.concatenate(bordas))

    return nos_contorno, valores_contorno

A formulação variacional discreta leva a um sistema linear da forma

$$
A U = b.
$$

As entradas da matriz de rigidez são dadas por

$$
A_{ij}
=
\int_\Omega
\nabla \varphi_j\cdot \nabla \varphi_i
\,dx.
$$

O vetor do lado direito é dado por

$$
b_i
=
\int_\Omega
f\varphi_i
\,dx.
$$

Como a integração é feita elemento a elemento, calculamos as contribuições locais

$$
A_{ij}^{(K)}
=
\int_K
\nabla \varphi_j\cdot \nabla \varphi_i
\,dx,
$$

e

$$
b_i^{(K)}
=
\int_K
f\varphi_i
\,dx.
$$

Depois, somamos essas contribuições na matriz e no vetor globais.

As funções de forma são definidas inicialmente no elemento de referência $\hat K$.

Para calcular a matriz de rigidez no elemento físico $K$, precisamos dos gradientes em relação às coordenadas físicas $(x,y)$.

Pela regra da cadeia,

$$
\nabla_x \varphi_i
=
J_K^{-T}
\nabla_{\xi,\eta}\hat\varphi_i.
$$

Ou seja, se

$$
\nabla_{\xi,\eta}\hat\varphi_i
=
\begin{bmatrix}
\frac{\partial \hat\varphi_i}{\partial \xi}\\
\frac{\partial \hat\varphi_i}{\partial \eta}
\end{bmatrix},
$$

então

$$
\nabla_x \varphi_i
=
J_K^{-T}
\begin{bmatrix}
\frac{\partial \hat\varphi_i}{\partial \xi}\\
\frac{\partial \hat\varphi_i}{\partial \eta}
\end{bmatrix}.
$$

Com isso, a matriz local fica

$$
A_{ij}^{(K)}
\approx
\sum_{q}
w_q
\left(
\nabla_x\varphi_j(x_q)\cdot\nabla_x\varphi_i(x_q)
\right)
|\det J_K(\xi_q,\eta_q)|.
$$

In [43]:
def montar_sistema_poisson(
    coordenadas: np.ndarray,
    elementos: np.ndarray,
) -> tuple:
    numero_nos = coordenadas.shape[0]

    matriz_global = lil_matrix((numero_nos, numero_nos))
    vetor_global = np.zeros(numero_nos)

    pontos_gauss, pesos_gauss = np.polynomial.legendre.leggauss(3)

    for elemento in elementos:
        vertices = coordenadas[elemento]

        matriz_local = np.zeros((4, 4))
        vetor_local = np.zeros(4)

        for xi, peso_xi in zip(pontos_gauss, pesos_gauss):
            for eta, peso_eta in zip(pontos_gauss, pesos_gauss):
                peso = peso_xi * peso_eta

                N = funcoes_forma_bilineares(xi, eta)

                dN_dxi, dN_deta = derivadas_funcoes_forma_bilineares(xi, eta)
                derivadas_referencia = np.vstack([dN_dxi, dN_deta])

                jacobiano = calcular_jacobiano(xi, eta, vertices)
                det_jacobiano = abs(np.linalg.det(jacobiano))

                # gradientes físicos das funções de forma
                gradientes_fisicos = np.linalg.solve(
                    jacobiano.T,
                    derivadas_referencia,
                )

                x, y = avaliar_transformacao(xi, eta, vertices)
                valor_fonte = fonte(x, y)

                matriz_local += (
                    gradientes_fisicos.T
                    @ gradientes_fisicos
                    * det_jacobiano
                    * peso
                )

                vetor_local += (
                    N
                    * valor_fonte
                    * det_jacobiano
                    * peso
                )

        for i_local, i_global in enumerate(elemento):
            vetor_global[i_global] += vetor_local[i_local]

            for j_local, j_global in enumerate(elemento):
                matriz_global[i_global, j_global] += matriz_local[i_local, j_local]

    return matriz_global.tocsr(), vetor_global

A condição de contorno é essencial, isto é, deve ser imposta diretamente no espaço discreto.

Separamos os nós em dois conjuntos:

- nós de contorno, onde $u_h$ é conhecido;
- nós livres, onde $u_h$ é incógnita.

Escrevendo o sistema em blocos,

$$
\begin{bmatrix}
A_{LL} & A_{LC}\\
A_{CL} & A_{CC}
\end{bmatrix}
\begin{bmatrix}
U_L\\
U_C
\end{bmatrix}
=
\begin{bmatrix}
b_L\\
b_C
\end{bmatrix},
$$

onde $L$ representa os graus de liberdade livres e $C$ os graus de liberdade de contorno.

Como $U_C$ é conhecido pela condição de Dirichlet, resolvemos

$$
A_{LL}U_L
=
b_L-A_{LC}U_C.
$$


In [44]:
def resolver_poisson_dirichlet(
    coordenadas: np.ndarray,
    elementos: np.ndarray,
    bordas: list[np.ndarray],
) -> np.ndarray:
    matriz_global, vetor_global = montar_sistema_poisson(coordenadas, elementos)

    nos_contorno, valores_contorno = construir_valores_contorno(coordenadas, bordas)

    numero_nos = coordenadas.shape[0]
    todos_os_nos = np.arange(numero_nos)

    nos_livres = np.setdiff1d(todos_os_nos, nos_contorno)

    solucao = np.zeros(numero_nos)
    solucao[nos_contorno] = valores_contorno[nos_contorno]

    matriz_livre = matriz_global[nos_livres][:, nos_livres]
    vetor_livre = (
        vetor_global[nos_livres]
        -
        matriz_global[nos_livres][:, nos_contorno] @ solucao[nos_contorno]
    )

    solucao[nos_livres] = spsolve(matriz_livre, vetor_livre)

    return solucao

Depois de calcular $u_h$, queremos determinar a média da solução aproximada em $\Omega$:

$$
\frac{1}{|\Omega|}
\int_\Omega u_h(x)\,dx.
$$

A integral é calculada com a mesma quadratura de Gauss $3\times 3$ usada no item (b).

Em cada elemento $K$, temos

$$
u_h(x,y)
=
\sum_{i=1}^{4} U_i^{(K)}\varphi_i(x,y),
$$

ou, no elemento de referência,

$$
u_h(F_K(\xi,\eta))
=
\sum_{i=1}^{4} U_i^{(K)}\hat\varphi_i(\xi,\eta).
$$

Assim,

$$
\int_\Omega u_h(x)\,dx
=
\sum_{K\in\tau_h}
\int_{\hat K}
u_h(F_K(\xi,\eta))
|\det J_K(\xi,\eta)|
\,d\xi\,d\eta.
$$

A área $|\Omega|$ é calculada por

$$
|\Omega|
=
\int_\Omega 1\,dx.
$$

In [45]:
def integrar_solucao_na_malha(
    solucao: np.ndarray,
    coordenadas: np.ndarray,
    elementos: np.ndarray,
) -> tuple[float, float, float]:
    pontos_gauss, pesos_gauss = np.polynomial.legendre.leggauss(3)

    integral_solucao = 0.0
    area = 0.0

    for elemento in elementos:
        vertices = coordenadas[elemento]
        valores_locais = solucao[elemento]

        for xi, peso_xi in zip(pontos_gauss, pesos_gauss):
            for eta, peso_eta in zip(pontos_gauss, pesos_gauss):
                peso = peso_xi * peso_eta

                N = funcoes_forma_bilineares(xi, eta)
                jacobiano = calcular_jacobiano(xi, eta, vertices)

                det_jacobiano = abs(np.linalg.det(jacobiano))

                uh = N @ valores_locais

                integral_solucao += uh * det_jacobiano * peso
                area += det_jacobiano * peso

    media = integral_solucao / area

    return integral_solucao, area, media

In [46]:
solucao = resolver_poisson_dirichlet(
    coordenadas=coordenadas,
    elementos=elementos,
    bordas=bordas,
)

integral_uh, area_omega, media_uh = integrar_solucao_na_malha(
    solucao=solucao,
    coordenadas=coordenadas,
    elementos=elementos,
)

print(f"Área de Ω: {area_omega:.6f}")
print(f"Integral de uh em Ω: {integral_uh:.6f}")
print(f"Média de uh em Ω: {media_uh:.6f}")

Área de Ω: 4606.110419
Integral de uh em Ω: 79692.760042
Média de uh em Ω: 17.301531
